# Annular Boussinesq — AWFNO JCP one-run numerical study

Google Colab GPU: select **Runtime → Run all**. This executes two suites sequentially and stores checkpoints, plots, and results on Google Drive so completed trials can be resumed. During the original stage, each training-set size is now trained, evaluated and written to CSV/JSON *before* proceeding to the next size. Live subprocess training logs are streamed into Colab.

1. **Original sample-size sweep:** TB-AWFNO, AWFNO, WFNO, full-trajectory FNO at N = 25, 50, 100, 200, 400.
2. **Controlled reviewer suites (three seeds):** LGL / LLG / GLL orderings, LLL / GGG component ablations, window K = 1, 10, 30, radius r = 1, 2, 3, Fourier modes (8,12)/(16,24)/(24,32), parameter-matched WFNO and temporal-basis head.

The observable is the scalar HyTeG annulus field `c_grid`, not velocity or pressure. This notebook does **not** implement modern F-FNO, U-Net, CNO, or U-NO baselines. The controlled study has 51 fits plus up to 20 original fits, so it may require more than one Colab session. Re-executing skips saved completed trials.

In [ ]:
#@title 1. Mount Drive and check annulus data
from google.colab import drive
from pathlib import Path
import torch, os, sys, subprocess
drive.mount('/content/gdrive')
DATASET=Path('/content/gdrive/MyDrive/anus/hyteg_annulus_c_grid_64x128_samples500.pt')
OUTPUT_ROOT=Path('/content/gdrive/MyDrive/anus/Boussinesq_JCP')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
assert DATASET.is_file(), f"Missing dataset: {DATASET}"
assert torch.cuda.is_available(), "Set Google Colab runtime to GPU."
print('GPU:',torch.cuda.get_device_name(0))
print('Data:',DATASET,'size GiB:',round(DATASET.stat().st_size/1024**3,3))
print('Outputs:',OUTPUT_ROOT)

In [ ]:
#@title 2. Install the reference full-trajectory FNO dependency
subprocess.check_call([sys.executable,'-m','pip','-q','install','neuraloperator==2.0.0'])
import importlib.metadata
print('NeuralOperator:',importlib.metadata.version('neuraloperator'))

## Experiment driver

The full experiment implementation is available as [readable Python source](annulus2d_jcp_experiments.py) in this repository. The next cell downloads that file directly from GitHub and checks its SHA-256 before executing it. There is no embedded or obfuscated code.


In [ ]:
#@title 3. Load the readable experiment driver from GitHub
from pathlib import Path
from urllib.request import urlopen
import hashlib

# Full, readable implementation:
# https://github.com/dimigourz/AWFNO/blob/main/Boussinesq2D/annulus2d_jcp_experiments.py
SOURCE_URL = (
    "https://raw.githubusercontent.com/dimigourz/AWFNO/main/"
    "Boussinesq2D/annulus2d_jcp_experiments.py"
)
EXPECTED_SHA256 = "a4d78c78bda2703e408ae90ed501dd56334294c9082c086ab2516df6a7acc883"
DRIVER = Path("/content/annulus2d_jcp_experiments.py")

with urlopen(SOURCE_URL, timeout=60) as response:
    source = response.read()
actual_sha256 = hashlib.sha256(source).hexdigest()
if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        f"Experiment source checksum differs from validated version: {actual_sha256}. "
        "Inspect the GitHub source and deliberately update EXPECTED_SHA256 before continuing."
    )

DRIVER.write_bytes(source)
print(f"Experiment script ready: {DRIVER} ({len(source):,} bytes)")

In [ ]:
#@title 4. Configuration — full original + controlled runs
RUN_ORIGINAL = True #@param {type:'boolean'}
RUN_CONTROLLED = True #@param {type:'boolean'}
CONTROLLED_SUITE = 'all' #@param ['all','ordering','components','window','radius','modes','matched','temporal_basis']
SEEDS = '1234,1235,1236' #@param {type:'string'}
SAMPLE_SIZES = '25,50,100,200,400' #@param {type:'string'}
N_TRAIN_CONTROLLED = 400 #@param {type:'integer'}
EPOCHS = 150 #@param {type:'integer'}
BATCH_SIZE = 32 #@param {type:'integer'}
FORCE_RETRAIN = False #@param {type:'boolean'}
WINDOW_SWEEP = '1,10,30'
RADIUS_SWEEP = '1,2,3'
assert RUN_ORIGINAL or RUN_CONTROLLED
print('Original:',RUN_ORIGINAL,'Controlled:',RUN_CONTROLLED,'Seeds:',SEEDS)

In [ ]:
#@title 5. Run each selected suite (Drive checkpoints allow resumption)
cfg=os.environ.copy()
cfg.update({
  'DATA_PATH':str(DATASET),'OUTPUT_ROOT':str(OUTPUT_ROOT),
  'RUN_SAMPLE_SIZES':SAMPLE_SIZES,
  'EPOCHS':str(EPOCHS),'BATCH_SIZE':str(BATCH_SIZE),
  'SPLIT_MODE':'contiguous','SPLIT_SEED':'2026',
  'N_VAL':'30','N_TEST':'70',
  'WINDOW_K':'30','WINDOW_STRIDE':'30','N_TIME_MODES':'18',
  'WIDTH':'32','N_LAYERS':'4','MODES_R':'16','MODES_THETA':'24',
  'LOCAL_RADIUS':'2',
  'REBUTTAL_N_TRAIN':str(N_TRAIN_CONTROLLED),
  'REBUTTAL_SEEDS':SEEDS,'REBUTTAL_RUN':CONTROLLED_SUITE,
  'REBUTTAL_OUTPUT':str(OUTPUT_ROOT/'jcp_controlled_annulus'),
  'WINDOW_SWEEP':WINDOW_SWEEP,'RADIUS_SWEEP':RADIUS_SWEEP,
  'FORCE_RETRAIN':'1' if FORCE_RETRAIN else '0','PYTHONUNBUFFERED':'1'
})
# Stream child process logs to Colab and save them on Drive.
# A heartbeat continues even during slow model-library imports or data loading.
from threading import Event, Thread
from IPython.display import display, Image, Markdown

def preview_saved_plots():
    # Show the newest sample-size and per-time error plots in Colab.
    patterns = ['sample_size_sweep_abs_full_L2*.png',
                'sample_size_sweep_abs_final_L2*.png',
                'all_tests_mean_median_L2_error_*.png',
                '*_annulus_fields*.png']
    for pat in patterns:
        candidates=list(OUTPUT_ROOT.rglob(pat))
        if candidates:
            figure=max(candidates,key=lambda p:p.stat().st_mtime)
            print('Plot:',figure,flush=True)
            display(Image(filename=str(figure), width=850))

for stage, enabled in [('original', RUN_ORIGINAL), ('rebuttal', RUN_CONTROLLED)]:
    if not enabled:
        continue
    env=dict(cfg)
    env['EXPERIMENT_SUITE']=stage
    log_path=OUTPUT_ROOT / f'{stage}_stdout.log'
    print(f'\\nSTART: {stage} | driver={DRIVER} | output={OUTPUT_ROOT}',flush=True)
    print(f'Live training output will appear below. Full log: {log_path}',flush=True)
    done=Event()

    def heartbeat():
        while not done.wait(45):
            print(f'[{stage}] subprocess is still running; checking for training output...',flush=True)

    watcher=Thread(target=heartbeat,daemon=True)
    watcher.start()
    try:
        with log_path.open('a',encoding='utf-8') as log:
            with subprocess.Popen(
                [sys.executable,'-u',str(DRIVER)],
                env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                text=True,bufsize=1
            ) as proc:
                for line in proc.stdout:
                    print(line,end='',flush=True)
                    log.write(line)
                    log.flush()
                exit_status=proc.wait()
            if exit_status != 0:
                raise subprocess.CalledProcessError(exit_status,[sys.executable,str(DRIVER)])
    finally:
        done.set()
        watcher.join()
    print('DONE:',stage,flush=True)
    if stage == 'original':
        preview_saved_plots()
print('Finished all selected stages. Results:',OUTPUT_ROOT)

## Saved summaries

The final cell is safe to rerun independently after interrupted training. It displays completed summaries only and calculates mean/standard deviation across available controlled seeds (not across sample sizes).

In [ ]:
#@title 6. Aggregate available experimental results
import json
import pandas as pd
from IPython.display import display
for source in sorted(OUTPUT_ROOT.rglob('sample_size_sweep_summary_run_*.json')):
    print('Original sample-size results:',source)
    display(pd.DataFrame(json.loads(source.read_text())))
p=OUTPUT_ROOT/'jcp_controlled_annulus'/'results.csv'
if p.exists():
    raw=pd.read_csv(p)
    print('Controlled trials:',len(raw))
    display(raw.head(10))
    metrics=[x for x in ['full_l2_mean','final_l2_mean','low_theta_l2','high_theta_l2','sharp_region_l2','train_seconds','rollout_seconds','numel_parameters','real_scalar_dof','peak_rollout_gpu_MiB'] if x in raw.columns]
    agg=raw.groupby('experiment',as_index=False)[metrics].agg(['mean','std'])
    agg.columns=['_'.join(filter(None,map(str,col))) for col in agg.columns.to_flat_index()]
    out=p.parent/'results_aggregated_by_seed.csv'
    agg.to_csv(out,index=False)
    print('Aggregated:',out)
    display(agg)
else:
    print('Controlled trials not started:',p)

In [ ]:
#@title 7. View already saved figures (safe after stopping an interrupted run)
from pathlib import Path
from IPython.display import display, Image
root=Path('/content/gdrive/MyDrive/anus/Boussinesq_JCP')
patterns=['sample_size_sweep_abs_full_L2*.png',
          'sample_size_sweep_abs_final_L2*.png',
          'all_tests_mean_median_L2_error_*.png',
          '*_annulus_fields*.png',
          'best_1_test_*.png',
          'worst_1_test_*.png']
for pattern in patterns:
    matches=list(root.rglob(pattern))
    if matches:
        p=max(matches,key=lambda p:p.stat().st_mtime)
        print('Saved plot:',p)
        display(Image(filename=str(p),width=850))
    else:
        print('Not yet saved:',pattern)
print('All PNG files:',len(list(root.rglob('*.png'))))